# Week 62, Day 1: Bayesian networks: joint distributions from local pieces

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w062/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import itertools
nodes = ["S", "A", "U", "R", "P", "B"]
parents = {"S": [], "A": [], "U": [], "R": ["S", "A", "U"], "P": ["U"], "B": ["R"]}
# probability that each node is 1, given its parents' values (in the order listed)
cpt = {
    "S": {(): 0.2}, "A": {(): 0.5}, "U": {(): 0.3},
    "R": {(s, a, u): min(0.97, 0.05 + 0.55 * s + 0.15 * s * a + 0.5 * u) for s in (0, 1) for a in (0, 1) for u in (0, 1)},
    "P": {(0,): 0.1, (1,): 0.85},
    "B": {(0,): 0.1, (1,): 0.8},
}
print("P(R = 1 | S, A, U):", {k: round(v, 2) for k, v in cpt["R"].items()})

### Exercise: The joint as a product

Write joint(x), the probability of a full assignment (a dict from node name to 0 or 1), as the product of each node's conditional probability given its parents. Then count the free parameters of the network: one per row of each conditional table.

In [ ]:
def joint(x):
    p = 1.0
    for v in nodes:
        p1 = cpt[v][tuple(x[u] for u in parents[v])]
        p *= ...
    return p

n_params = ...
print(f"the network needs {n_params} numbers; the full joint over 6 binary variables needs {2 ** 6 - 1}")

In [ ]:
import numpy as np
def test_sum():
    "the joint sums to one over all 64 configurations"
    assert abs(sum(joint(dict(zip(nodes, x))) for x in itertools.product((0, 1), repeat=6)) - 1) < 1e-12
def test_count():
    "the network needs 15 numbers, against 63 for the full joint"
    assert n_params == 15

test_sum()
test_count()
print("All checks passed")

### Exercise: Posteriors by summing the joint

Write posterior(query, evidence), which enumerates all 64 configurations, keeps those consistent with the evidence, and returns the probability that the query node is 1. Compute P(S = 1 | B = 1) and P(S = 1 | R = 1).

In [ ]:
def posterior(query, evidence):
    num = den = 0.0
    for vals in itertools.product((0, 1), repeat=len(nodes)):
        x = dict(zip(nodes, vals))
        if any(x[k] != v for k, v in evidence.items()):
            continue
        p = joint(x)
        den += p
        num += ...
    return num / den

post_B = posterior("S", {"B": 1})
post_R = posterior("S", {"R": 1})
print(f"P(S = 1) = 0.200; P(S = 1 | report 'seen') = {post_B:.3f}; P(S = 1 | neuron fired) = {post_R:.3f}")

In [ ]:
import numpy as np
def test_post():
    "posterior returns P(query = 1 | evidence) and agrees with the prior when there is no evidence"
    assert abs(posterior("S", {}) - 0.2) < 1e-12
def test_report():
    "a seen report roughly doubles the probability that the stimulus was present"
    assert post_B > 0.3

test_post()
test_report()
print("All checks passed")

### Exercise: The pupil explains the response away

Compute P(S = 1 | R = 1, P = 1) and P(S = 1 | R = 1, P = 0), and compare them with P(S = 1 | R = 1).

In [ ]:
post_RP1 = ...
post_RP0 = ...
print(f"P(S | neuron fired) = {post_R:.3f}; with a dilated pupil {post_RP1:.3f}; with a constricted pupil {post_RP0:.3f}")

In [ ]:
import numpy as np
def test_away():
    "learning that the pupil was dilated lowers the probability of the stimulus given that the neuron fired; a constricted pupil raises it"
    assert post_RP1 < post_R < post_RP0
def test_independent():
    "without observing the neuron, the pupil says nothing about the stimulus"
    assert abs(posterior("S", {"P": 1}) - 0.2) < 1e-12

test_away()
test_independent()
print("All checks passed")

### Exercise: Simulating trials and rejecting

Write ancestral_sample(n, g), which samples n trials node by node in the order of the nodes list (already topological), using vectorised lookups into the conditional tables. Estimate P(S = 1 | R = 1, P = 1) from the trials that match the evidence, and report how many were kept.

In [ ]:
def ancestral_sample(n, g):
    out = {}
    for v in nodes:
        pa = parents[v]
        p1 = np.array([cpt[v][tuple(int(out[u][i]) for u in pa)] for i in range(n)]) if pa else np.full(n, cpt[v][()])
        out[v] = ...
    return out

samples = ancestral_sample(20000, np.random.default_rng(621))
keep = (samples["R"] == 1) & (samples["P"] == 1)
est = samples["S"][keep].mean()
print(f"kept {keep.sum()} of 20000 samples; estimate {est:.3f}, exact {post_RP1:.3f}")

In [ ]:
import numpy as np
def test_marginal():
    "the sampled frequency of reports matches the exact marginal to within 0.01"
    assert abs(samples["B"].mean() - posterior("B", {})) < 0.01
def test_reject():
    "the rejection estimate of P(S = 1 | R = 1, P = 1) is within 0.03 of the exact value"
    assert abs(est - post_RP1) < 0.03

test_marginal()
test_reject()
print("All checks passed")